# DepthwiseCNN — FFT-75 Benchmark (Kaggle)

End-to-end notebook for training and evaluating the **DepthwiseCNN** model family
on the FFT-75 file-fragment classification benchmark.

Reference paper: *File Fragment Type Classification Using Light-Weight Convolutional Neural Networks*

---

## Fragment-size / variant switch

**Everything** is driven by the single config cell below.  To switch scenarios:

| Change | Where |
|--------|-------|
| 512-byte ↔ 4096-byte | `FRAGMENT_SIZE` in **§ 0** |
| `dsc` / `dsc-se` / `m-dsc` | `VARIANT` in **§ 0** |

No other cell needs editing.

---

## Notebook structure

| § | Description |
|---|-------------|
| 0 | Runtime config (the only cell you need to edit) |
| 1 | Environment setup |
| 2 | FFT-75 dataset download |
| 3 | Dataset validation |
| 4 | Sanity training pass (5 epochs, tiny subset) |
| 5 | Full training |
| 6 | Evaluation (test split) |
| 7 | Results summary |

---
## § 0 — Runtime configuration (edit here only)

Change `FRAGMENT_SIZE` and `VARIANT` to run a different benchmark scenario.
All other cells read from these variables.

In [ ]:
# =============================================================================
# § 0  RUNTIME CONFIGURATION
# =============================================================================

# Fragment size in bytes — 512 or 4096
FRAGMENT_SIZE = 4096   # <-- change to 512 for the 512-byte scenario

# Architecture variant — 'dsc' | 'dsc-se' | 'm-dsc'
VARIANT = 'dsc'        # <-- change to select a different model variant

# Training hyperparameters
EPOCHS      = 30
BATCH_SIZE  = 256
LR          = 1e-3
PATIENCE    = 5        # early-stopping patience
SEED        = 42

# Sanity-pass settings (§ 4)
SANITY_EPOCHS  = 5
SANITY_SAMPLES = 2000  # samples per split for the sanity pass

print(f'Config locked: fragment_size={FRAGMENT_SIZE}, variant={VARIANT}')

---
## § 1 — Environment setup

In [ ]:
import os, sys

# ---------------------------------------------------------------------------
# Kaggle path layout
# ---------------------------------------------------------------------------
WORKING_DIR  = '/kaggle/working'
REPO_DIR     = os.path.join(WORKING_DIR, 'deepcarv')   # cloned repo
DATA_ROOT    = os.path.join(WORKING_DIR, 'data')
FFT75_DIR    = os.path.join(DATA_ROOT,   'FFT-75')
OUTPUTS_DIR  = os.path.join(WORKING_DIR, 'outputs', 'DepthwiseCNN')
CKPT_DIR     = os.path.join(OUTPUTS_DIR)
EVAL_DIR     = os.path.join(OUTPUTS_DIR, 'eval')
LOG_DIR      = os.path.join(WORKING_DIR, 'logs')
CONFIG_PATH  = os.path.join(
    REPO_DIR, 'benchmarks', 'DepthwiseCNN', 'configs', 'benchmark.yaml'
)

for d in [DATA_ROOT, FFT75_DIR, OUTPUTS_DIR, EVAL_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# Make repo importable without installing it
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Tell DeepCarv's path resolver where data and outputs live
os.environ['DEEPCARV_DATA_ROOT']    = DATA_ROOT
os.environ['DEEPCARV_OUTPUTS_DIR']  = os.path.join(WORKING_DIR, 'outputs')
os.environ['DEEPCARV_CHECKPOINTS_DIR'] = CKPT_DIR
os.environ['DEEPCARV_LOGS_DIR']     = LOG_DIR
os.environ['KAGGLE_RUNTIME']        = '1'

print('Working dir :', WORKING_DIR)
print('Repo dir    :', REPO_DIR)
print('FFT-75 dir  :', FFT75_DIR)
print('Outputs dir :', OUTPUTS_DIR)

In [ ]:
# Install / verify dependencies
!pip install -q pyyaml torch torchvision timm scikit-learn pandas matplotlib seaborn tqdm

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Clone (or update) the DeepCarv repository
import subprocess

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    result = subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/<your-org>/deepcarv.git',  # <-- update URL
         REPO_DIR],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
else:
    result = subprocess.run(
        ['git', '-C', REPO_DIR, 'pull', '--ff-only'],
        capture_output=True, text=True
    )
    print('Repo already present, pulled latest:', result.stdout.strip())

---
## § 2 — FFT-75 dataset download

The FFT-75 dataset must be available at `FFT75_DIR` with the layout:
```
FFT-75/
├── 512/
│   ├── train.npz
│   ├── val.npz
│   └── test.npz
└── 4096/
    ├── train.npz
    ├── val.npz
    └── test.npz
```
Download via Kaggle dataset attachment (preferred) or the `gdown` cell below.

In [ ]:
# ---------------------------------------------------------------------------
# Option A: Kaggle dataset attachment
# ---------------------------------------------------------------------------
# If the FFT-75 dataset is attached to this notebook as a Kaggle input dataset,
# copy or symlink it into FFT75_DIR.

KAGGLE_INPUT = '/kaggle/input'
# Update the dataset folder name to match the attached input.
KAGGLE_DATASET_FOLDER = 'fft-75-npz'   # <-- update if your dataset has a different name
kaggle_src = os.path.join(KAGGLE_INPUT, KAGGLE_DATASET_FOLDER)

if os.path.isdir(kaggle_src):
    import shutil
    if not os.path.isdir(os.path.join(FFT75_DIR, str(FRAGMENT_SIZE))):
        shutil.copytree(kaggle_src, FFT75_DIR, dirs_exist_ok=True)
    print('Dataset mounted from Kaggle input:', kaggle_src)
else:
    print('Kaggle input not found — falling through to Option B (gdown).')

In [ ]:
# ---------------------------------------------------------------------------
# Option B: Google Drive download via gdown
# ---------------------------------------------------------------------------
# Fill in the Google Drive file IDs for the FFT-75 archive(s).
# Skip this cell if Option A succeeded.

import os

GDRIVE_FILE_IDS = {
    # 'fft75_512.zip':  '<google-drive-id-for-512-archive>',
    # 'fft75_4096.zip': '<google-drive-id-for-4096-archive>',
}

already_have = os.path.isdir(os.path.join(FFT75_DIR, str(FRAGMENT_SIZE)))

if already_have:
    print(f'FFT-75/{FRAGMENT_SIZE}/ already present — skipping download.')
elif GDRIVE_FILE_IDS:
    !pip install -q gdown
    import gdown, zipfile, tempfile

    for fname, fid in GDRIVE_FILE_IDS.items():
        dest = os.path.join(WORKING_DIR, fname)
        if not os.path.exists(dest):
            print(f'Downloading {fname} …')
            gdown.download(id=fid, output=dest, quiet=False)
        print(f'Extracting {fname} …')
        with zipfile.ZipFile(dest, 'r') as z:
            z.extractall(FFT75_DIR)
    print('Download and extraction complete.')
else:
    print(
        'WARNING: Neither Kaggle input nor gdown IDs found.\n'
        'Provide the FFT-75 dataset at:', FFT75_DIR
    )

---
## § 3 — Dataset validation

Verify that the NPZ files are present, correctly keyed (`x`, `y`), and have
the right shapes before any training begins.

In [ ]:
import numpy as np
from pathlib import Path

def validate_npz(path: str, frag_size: int) -> None:
    p = Path(path)
    assert p.exists(), f'Missing: {p}'
    data = np.load(str(p))
    assert set(data.files) == {'x', 'y'}, (
        f'{p.name}: expected keys {{x, y}}, got {set(data.files)}'
    )
    X, y = data['x'], data['y']
    assert X.ndim == 2 and X.shape[1] == frag_size, (
        f'{p.name}: X shape {X.shape}, expected (N, {frag_size})'
    )
    assert y.ndim == 1 and len(y) == len(X), (
        f'{p.name}: y shape {y.shape}, len(X)={len(X)}'
    )
    num_classes = int(y.max()) + 1
    print(
        f'  OK  {p.name:<12}  N={len(X):>8,}  '
        f'classes={num_classes}  '
        f'X.dtype={X.dtype}  y.dtype={y.dtype}'
    )

print(f'Validating FFT-75 / {FRAGMENT_SIZE}-byte splits …')
fft75_frag_dir = os.path.join(FFT75_DIR, str(FRAGMENT_SIZE))

for split in ('train', 'val', 'test'):
    validate_npz(os.path.join(fft75_frag_dir, f'{split}.npz'), FRAGMENT_SIZE)

print('\nDataset validation passed. ✓')

---
## § 4 — Sanity training pass

Train for 5 epochs on 2 000 samples per split to confirm the pipeline is
end-to-end working before launching the full run.

In [ ]:
# Write a temporary config for the sanity pass
import yaml, copy

with open(CONFIG_PATH) as f:
    sanity_cfg = yaml.safe_load(f)

# Override for sanity
sanity_cfg['dataset']['root_dir']      = FFT75_DIR
sanity_cfg['dataset']['fragment_size'] = FRAGMENT_SIZE
sanity_cfg['model']['kwargs']['variant'] = VARIANT
sanity_cfg['training']['epochs']       = SANITY_EPOCHS
sanity_cfg['training']['batch_size']   = BATCH_SIZE
sanity_cfg['training']['seed']         = SEED
sanity_cfg['training']['device']       = 'cuda' if torch.cuda.is_available() else 'cpu'
sanity_cfg['paths']['run_outputs']     = os.path.join(OUTPUTS_DIR, 'sanity')

SANITY_CONFIG = os.path.join(WORKING_DIR, 'sanity_config.yaml')
with open(SANITY_CONFIG, 'w') as f:
    yaml.safe_dump(sanity_cfg, f, sort_keys=False)

print('Sanity config written to:', SANITY_CONFIG)
print(f'  fragment_size={FRAGMENT_SIZE}  variant={VARIANT}  epochs={SANITY_EPOCHS}')

In [ ]:
# Run sanity pass using a tiny subset (inline, no subprocess, for fast feedback)
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', force=True)

from src.data.dataset import FragmentDataset, build_dataloader
from src.models.registry import build_model
from src.training.trainer import Trainer, TrainerConfig
from src.utils.seed import set_seed
from pathlib import Path

set_seed(SEED)

train_ds_sanity = FragmentDataset(
    root_dir=FFT75_DIR, split='train',
    fragment_size=FRAGMENT_SIZE, cache=True,
    tiny_subset=SANITY_SAMPLES,
)
val_ds_sanity = FragmentDataset(
    root_dir=FFT75_DIR, split='val',
    fragment_size=FRAGMENT_SIZE, cache=True,
    tiny_subset=SANITY_SAMPLES,
)

sanity_loader_tr = build_dataloader(train_ds_sanity, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
sanity_loader_va = build_dataloader(val_ds_sanity,   batch_size=BATCH_SIZE, shuffle=False)

sanity_model = build_model('depthwisecnn', num_classes=train_ds_sanity.num_classes, variant=VARIANT)
print(f'Model: {sanity_model.name}  params={sanity_model.num_parameters():,}')

sanity_tcfg = TrainerConfig(
    epochs=SANITY_EPOCHS, lr=LR, batch_size=BATCH_SIZE,
    patience=SANITY_EPOCHS,   # no early stopping during sanity
    device='cuda' if torch.cuda.is_available() else 'cpu',
    amp=torch.cuda.is_available(),
    seed=SEED,
)
sanity_trainer = Trainer(
    sanity_model, sanity_tcfg,
    run_dir=Path(OUTPUTS_DIR) / 'sanity',
)

sanity_history = sanity_trainer.fit(sanity_loader_tr, sanity_loader_va)
print(f'Sanity pass complete. Final val_acc={sanity_history.val_acc[-1]:.4f}')
print('Pipeline is end-to-end healthy. ✓')

---
## § 5 — Full training

Launch full training via the benchmark script.  Uses the config in § 0.

In [ ]:
# Write the full-training config (full dataset, all epochs)
with open(CONFIG_PATH) as f:
    full_cfg = yaml.safe_load(f)

full_cfg['dataset']['root_dir']       = FFT75_DIR
full_cfg['dataset']['fragment_size']  = FRAGMENT_SIZE
full_cfg['model']['kwargs']['variant'] = VARIANT
full_cfg['training']['epochs']        = EPOCHS
full_cfg['training']['batch_size']    = BATCH_SIZE
full_cfg['training']['lr']            = LR
full_cfg['training']['patience']      = PATIENCE
full_cfg['training']['seed']          = SEED
full_cfg['training']['device']        = 'cuda' if torch.cuda.is_available() else 'cpu'
full_cfg['paths']['run_outputs']      = OUTPUTS_DIR
full_cfg['paths']['best_checkpoint']  = os.path.join(OUTPUTS_DIR, 'checkpoint_best.pt')
full_cfg['paths']['last_checkpoint']  = os.path.join(OUTPUTS_DIR, 'checkpoint_last.pt')
full_cfg['paths']['eval_outputs']     = EVAL_DIR

FULL_CONFIG = os.path.join(WORKING_DIR, 'full_config.yaml')
with open(FULL_CONFIG, 'w') as f:
    yaml.safe_dump(full_cfg, f, sort_keys=False)

print('Full-training config written to:', FULL_CONFIG)
print(f'  fragment_size={FRAGMENT_SIZE}  variant={VARIANT}  epochs={EPOCHS}')

In [ ]:
# Run full training inline so Kaggle output cells capture progress
from pathlib import Path
from benchmarks.DepthwiseCNN.scripts.train import main as train_main

train_main([
    '--config', FULL_CONFIG,
    '--run-name', f'depthwisecnn_{VARIANT}_{FRAGMENT_SIZE}b',
])

---
## § 6 — Evaluation (test split)

Load the best checkpoint and run the full evaluation suite:
metrics, confusion matrix, per-class results, predictions.

In [ ]:
from benchmarks.DepthwiseCNN.scripts.evaluate import main as eval_main

BEST_CKPT = os.path.join(OUTPUTS_DIR, 'checkpoint_best.pt')

eval_main([
    '--config',     FULL_CONFIG,
    '--checkpoint', BEST_CKPT,
    '--split',      'test',
    '--out-dir',    EVAL_DIR,
])

print('Evaluation complete. Outputs in:', EVAL_DIR)

---
## § 7 — Results summary

In [ ]:
import json
import pandas as pd

# ── Core metrics ─────────────────────────────────────────────────────────────
metrics_path = os.path.join(EVAL_DIR, 'metrics.json')
summary_path = os.path.join(EVAL_DIR, 'summary.json')

if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print('=== Evaluation Summary ===')
    for k, v in summary.items():
        if isinstance(v, float):
            print(f'  {k:<30} {v:.6f}')
        else:
            print(f'  {k:<30} {v}')
else:
    print('summary.json not found — check evaluation output dir:', EVAL_DIR)

In [ ]:
# ── Per-class metrics (top / bottom 10 by F1) ─────────────────────────────────
per_class_path = os.path.join(EVAL_DIR, 'per_class_metrics.csv')
if os.path.exists(per_class_path):
    per_class = pd.read_csv(per_class_path, index_col='class')
    print('\n=== Top 10 classes by F1 ===')
    print(per_class.sort_values('f1', ascending=False).head(10).to_string())
    print('\n=== Bottom 10 classes by F1 ===')
    print(per_class.sort_values('f1', ascending=True).head(10).to_string())

In [ ]:
# ── Confusion matrix heatmap ─────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')   # non-interactive backend for Kaggle
import matplotlib.pyplot as plt
import seaborn as sns

cm_path = os.path.join(EVAL_DIR, 'confusion_matrix.csv')
if os.path.exists(cm_path):
    cm = pd.read_csv(cm_path).values
    # Normalise per true class for readability
    cm_norm = cm.astype(float)
    row_sums = cm_norm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1          # avoid division by zero
    cm_norm = cm_norm / row_sums

    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        cm_norm, ax=ax, cmap='Blues',
        vmin=0, vmax=1,
        xticklabels=False, yticklabels=False,
        cbar_kws={'label': 'Fraction correct'},
    )
    ax.set_xlabel('Predicted class', fontsize=12)
    ax.set_ylabel('True class', fontsize=12)
    ax.set_title(
        f'DepthwiseCNN ({VARIANT}) — FFT-75 {FRAGMENT_SIZE}B\n'
        f'Normalised confusion matrix (test split)',
        fontsize=13,
    )
    fig.tight_layout()
    cm_fig_path = os.path.join(EVAL_DIR, 'confusion_matrix_heatmap.png')
    plt.savefig(cm_fig_path, dpi=150)
    plt.show()
    print('Confusion matrix saved to:', cm_fig_path)
else:
    print('confusion_matrix.csv not found — skipping heatmap.')

In [ ]:
# ── Final benchmark card ─────────────────────────────────────────────────────
from src.models.registry import build_model

model_info = build_model('depthwisecnn', num_classes=75, variant=VARIANT)
param_count = model_info.num_parameters()

print('\n' + '='*55)
print('  DepthwiseCNN Benchmark Card')
print('='*55)
print(f'  Variant         : {VARIANT}')
print(f'  Fragment size   : {FRAGMENT_SIZE} bytes')
print(f'  Parameters      : {param_count:,}')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        s = json.load(f)
    print(f'  Test accuracy   : {s.get("accuracy", "N/A"):.4f}')
    print(f'  Macro F1        : {s.get("macro_f1", "N/A"):.4f}')
    print(f'  Weighted F1     : {s.get("weighted_f1", "N/A"):.4f}')
    print(f'  Latency (ms/sa) : {s.get("latency_ms_per_sample", "N/A"):.3f}')
    print(f'  GPU             : {s.get("gpu_name", "N/A")}')
print('='*55)
print('\nOutputs saved to:', EVAL_DIR)